# 03 · Preprocesado común: oraciones, tokens y raíces

Tercer notebook del preprocesado (E2). Parte de los cuatro ficheros `*_limpio.jsonl` de los notebooks 01 y 02 y les aplica a todos **la misma función de preprocesado**:

1. Normalización Unicode (NFC).
2. División en oraciones.
3. Tokenización (`tokens`, para embeddings).
4. Raíces sin stopwords (`raices_bow`, para BoW / TF-IDF).

El resultado son los `*_procesado.jsonl`, con el formato de `ESQUEMA_DATOS.md`.

## 0. Preparación

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from nltk.tokenize import sent_tokenize

# El notebook se ejecuta desde la carpeta E2/, por eso las rutas empiezan por "..".
CARPETA_DATOS = Path("../data/processed")
FICHEROS = ["fallacyes_proto", "fallacyes_espont", "fallacyes_political", "congreso"]

corpus = {
    nombre: pd.read_json(CARPETA_DATOS / f"{nombre}_limpio.jsonl", lines=True, dtype=False)
    for nombre in FICHEROS
}
for nombre, tabla in corpus.items():
    print(f"{nombre}: {len(tabla)} textos")

fallacyes_proto: 3319 textos
fallacyes_espont: 1840 textos
fallacyes_political: 1951 textos
congreso: 2468 textos


## 1. Normalización Unicode (NFC)

Una letra con tilde se puede guardar de dos formas: como un solo carácter (`é`) o como la letra más una tilde suelta (`e` + `´`). Se ven igual, pero para Python son cadenas distintas, y el Word2Vec las trataría como dos palabras. NFC lo deja todo en la primera forma. **No quita las tildes.**

In [2]:
compuesta = "país"  # "i" + tilde suelta
print(compuesta, "==", "país", "→", compuesta == "país")
print("con NFC →", unicodedata.normalize("NFC", compuesta) == "país")

for nombre, tabla in corpus.items():
    cambian = tabla["texto"].map(lambda texto: unicodedata.normalize("NFC", texto) != texto)
    print(f"{nombre}: textos que cambian con NFC: {cambian.sum()}")

país == país → False
con NFC → True
fallacyes_proto: textos que cambian con NFC: 0
fallacyes_espont: textos que cambian con NFC: 0
fallacyes_political: textos que cambian con NFC: 0
congreso: textos que cambian con NFC: 0


En nuestros datos no cambia ningún texto. Se aplica igualmente: es una red de seguridad para textos nuevos (copiados de un PDF o escritos en un Mac) y es una línea de código.

## 2. División en oraciones: qué falla con `sent_tokenize`

`sent_tokenize` de NLTK usa el modelo Punkt para español. Punkt no corta en todos los puntos: tiene una lista de abreviaturas tras las que no corta. Se mira dónde se equivoca en nuestros datos.

In [3]:
pruebas = [
    "La Sra. García dijo que no. El Sr. López tampoco.",
    "Lo dijo el Prof. Martínez en clase.",
    "Vive en St. Louis desde hace años.",
    "Lo aprobó EE. UU. y luego Europa.",
    "Según el art. 155 de la Constitución.",
    "Vinieron todos: Juan, Ana, etc. Pero nadie habló.",
]
for prueba in pruebas:
    print(sent_tokenize(prueba, language="spanish"))

['La Sra.', 'García dijo que no.', 'El Sr. López tampoco.']
['Lo dijo el Prof.', 'Martínez en clase.']
['Vive en St.', 'Louis desde hace años.']
['Lo aprobó EE.', 'UU. y luego Europa.']
['Según el art.', '155 de la Constitución.']
['Vinieron todos: Juan, Ana, etc.', 'Pero nadie habló.']


In [4]:
# Cuántas veces sale cada abreviatura en cada corpus
ABREVIATURAS_A_BUSCAR = {
    "Sra.": r"\bSra\.", "Prof.": r"\bProf\.", "Dr. / Dra.": r"\bDra?\.", "St.": r"\bSt\.",
    "art.": r"\barts?\.", "núm.": r"\bnúm\.", "cap.": r"\bcap\.", "vol.": r"\bvol\.",
    "EE. UU.": r"EE\. ?UU\.", "etc.": r"\betc\.",
}
pd.DataFrame({
    nombre: {abreviatura: tabla["texto"].str.count(patron).sum() for abreviatura, patron in ABREVIATURAS_A_BUSCAR.items()}
    for nombre, tabla in corpus.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
Sra.,24,0,0,0
Prof.,8,0,0,0
Dr. / Dra.,12,0,0,0
St.,6,0,0,0
art.,0,0,1,0
núm.,0,0,0,1
cap.,0,0,0,4
vol.,0,0,0,3
EE. UU.,1,2,2,0
etc.,2,18,3,6


### 2.1 Oraciones muy largas

El texto de Political viene de una transcripción automática (WhisperX), que a veces puntúa poco. Se mira cuánto miden las oraciones de cada corpus.

In [5]:
oraciones_de_prueba = {
    nombre: tabla["texto"].map(lambda texto: sent_tokenize(texto, language="spanish")).explode()
    for nombre, tabla in corpus.items()
}
pd.DataFrame({
    nombre: {
        "oraciones": len(oraciones),
        "palabras (mediana)": oraciones.str.split().str.len().median(),
        "palabras (percentil 95)": oraciones.str.split().str.len().quantile(0.95),
        "palabras (máximo)": oraciones.str.split().str.len().max(),
        "más de 80 palabras": (oraciones.str.split().str.len() > 80).sum(),
    }
    for nombre, oraciones in oraciones_de_prueba.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
oraciones,5859.0,2745.0,4290.00,68449.0
palabras (mediana),13.0,16.0,16.00,20.0
palabras (percentil 95),30.0,38.0,44.55,61.0
palabras (máximo),90.0,83.0,191.00,252.0
más de 80 palabras,1.0,1.0,11.00,1125.0


In [6]:
# Ejemplos de oraciones de más de 80 palabras
for nombre in ["fallacyes_political", "congreso"]:
    oraciones = oraciones_de_prueba[nombre]
    largas = oraciones[oraciones.str.split().str.len() > 80]
    print(f"--- {nombre}")
    for oracion in largas.sample(2, random_state=1):
        print("   ", oracion[:350], "...\n")

--- fallacyes_political
    Que han bajado los porcentajes de gente que abandonaba pronto los estudios, los porcentajes de fracaso escolar, que le hemos dado un impulso muy importante a la formación profesional, que tenemos que adecuar lo que se estudie a lo que sea la realidad del mercado laboral, que vamos a crear 100.000 plazas, como le he señalado antes, en materia de for ...

    Yo creo que estas elecciones son decisivas porque si gobierna la derecha todo el mundo sabe que vamos a ir a peor y que se van a cuestionar derechos que hasta ahora estaban asentados y eso va a ocurrir si gobiernan PP, Ciudadanos y Vox y también si gobierna Ciudadanos con el Partido Socialista porque eso también es un gobierno de derechas y yo creo ...

--- congreso
    Aunque en nuestro país la Generalitat de Catalunya ya garantizara oficinas móviles para estos municipios, es a la banca a quien se tiene que imponer la obligatoriedad de garantizar este servicio y la gratuidad de hacer la retirada de efect

## 3. División en oraciones

**Decisión:** se usa Punkt (el modelo de `sent_tokenize`) añadiéndole las abreviaturas que le faltan, **solo las que salen en nuestros datos** (sección 2): *Sra., Dra., Prof., St., EE., art., núm., cap., vol.* Punkt guarda sus abreviaturas en minúscula y sin el punto final. Añadirlas es la forma habitual de adaptar Punkt a un dominio.

- *EE.* basta para *EE. UU.*: *UU.* ya estaba en la lista.
- *etc.* seguido de mayúscula se deja cortar: casi siempre la frase termina ahí (*"Juan, Ana, etc. Pero nadie habló."*).
- **Las oraciones largas no se parten** (sección 2.1): son frases largas de verdad, no errores. No molestan al BoW ni al Word2Vec, y la más larga (252 palabras) cabe en un transformer (unas 350 palabras como máximo). Lo que no cabe son las **intervenciones enteras** del Congreso (mediana de 394 palabras); con qué unidad se pasan por el detector se decidirá al usarlo. La jerarquía intervención → oración permite cualquier opción.

Se crea un tokenizador propio en vez de modificar el que usa `sent_tokenize` por dentro, para que el cambio quede a la vista.

In [7]:
from nltk.tokenize import PunktTokenizer

ABREVIATURAS_QUE_FALTAN = {"sra", "dra", "prof", "st", "ee", "art", "núm", "cap", "vol"}

tokenizador_de_oraciones = PunktTokenizer("spanish")
# _params es un atributo interno de NLTK, pero es la forma habitual de añadir abreviaturas a Punkt
tokenizador_de_oraciones._params.abbrev_types.update(ABREVIATURAS_QUE_FALTAN)


def dividir_en_oraciones(texto):
    return tokenizador_de_oraciones.tokenize(texto)


# Los casos de la sección 2, más alguno en el que sí hay que cortar
for prueba in pruebas + ["Lo aprobó EE. UU. Luego Europa.", "Lo dice el núm. 3. Y el cap. 2 también."]:
    print(dividir_en_oraciones(prueba))

['La Sra. García dijo que no.', 'El Sr. López tampoco.']
['Lo dijo el Prof. Martínez en clase.']
['Vive en St. Louis desde hace años.']
['Lo aprobó EE. UU. y luego Europa.']
['Según el art. 155 de la Constitución.']
['Vinieron todos: Juan, Ana, etc.', 'Pero nadie habló.']
['Lo aprobó EE. UU.', 'Luego Europa.']
['Lo dice el núm. 3.', 'Y el cap. 2 también.']


In [8]:
# Cuántas oraciones salen antes y después del cambio
oraciones_nuevas = {nombre: tabla["texto"].map(dividir_en_oraciones) for nombre, tabla in corpus.items()}
pd.DataFrame({
    nombre: {
        "oraciones antes": len(oraciones_de_prueba[nombre]),
        "oraciones ahora": oraciones_nuevas[nombre].str.len().sum(),
    }
    for nombre in corpus
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
oraciones antes,5859,2745,4290,68449
oraciones ahora,5819,2745,4289,68441


In [9]:
# Ejemplos de las prototípicas en los que ha cambiado la división
proto = corpus["fallacyes_proto"]["texto"]
antes = proto.map(lambda texto: sent_tokenize(texto, language="spanish"))
ahora = oraciones_nuevas["fallacyes_proto"]
for texto_antes, texto_ahora in list(zip(antes[antes != ahora], ahora[antes != ahora]))[:4]:
    print("antes:", texto_antes)
    print("ahora:", texto_ahora, "\n")

antes: ['La Sra.', 'Miller dio clases a tu hermano mayor, que era el payaso de la clase.', 'Decide ser estricta contigo desde el primer día de clase porque cree que tú también serás un payaso.']
ahora: ['La Sra. Miller dio clases a tu hermano mayor, que era el payaso de la clase.', 'Decide ser estricta contigo desde el primer día de clase porque cree que tú también serás un payaso.'] 

antes: ['La Sra.', 'Miller dará clases a ti y puede tener sus propias percepciones y expectativas sobre tu comportamiento en clase.', 'Esto no necesariamente significa que te tratará de manera diferente porque tu hermano mayor fue un payaso en su clase.']
ahora: ['La Sra. Miller dará clases a ti y puede tener sus propias percepciones y expectativas sobre tu comportamiento en clase.', 'Esto no necesariamente significa que te tratará de manera diferente porque tu hermano mayor fue un payaso en su clase.'] 

antes: ['Todos los profesores de escritura deben ser increíbles porque la Sra.', 'López lo es.']
aho